### Downstream Analysis

Post analysis after zonation trajectory inference:
- Differentially expressed genes / metabolites (Feature assotiation tests)
- Expression imputation via thin-plane spline(TPS) / cubic interpolation?
- Gene-Metabolite cross-correlations
- Gender comparison, etc.

In [ ]:
import os
import gc
import sys

import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq

import torch
import torch.nn as nn
import tifffile
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
from ipywidgets import interact, widgets
from IPython.display import display

from matplotlib import rcParams
rcParams['font.family'] = 'FreeSans'
rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
sys.path.append('..')
sys.path.append('../models/')
sys.path.append('../util')

import IO, utils, trajectory, configs, zonation, plot
import vgae, model_train
import scFates as scf

Load data & latent representations:

In [ ]:
xenium_path = '../data/xenium/'
desi_path = '../data/integrated/desi_xenium_map/'

sample_id = 'NIH_F5'
adata = IO.load_xenium(os.path.join(xenium_path, sample_id)) 
adata_desi = IO.load_desi(os.path.join(desi_path, sample_id+'.h5ad'), raw_img=False)
adata, adata_desi = IO.filter_cells(adata, adata_desi)

# Load latent representations
latent = np.load('../results/lynx_6_attn.npy')
n_latent = latent.shape[1]

adata.obsm['X_z'] = latent
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
sc.pp.neighbors(adata, use_rep='X_z')
sc.tl.umap(adata)

# Latent representation assignmeng for DESI
adata_desi.obsm['X_z'] = latent
adata_desi.obsm['X_umap'] = adata.obsm['X_umap']

Trajectory inference:

---

In [ ]:
dist_metric = 'euclidean'
n_nodes=10

# Xenium
trajectory.compute_trajectory(
    adata, 
    use_rep='X_z',
    n_nodes=n_nodes,
    dist_metric=dist_metric, 
)

# DESI
trajectory.compute_trajectory(
    adata_desi, 
    use_rep='X_z',
    n_nodes=n_nodes,
    dist_metric=dist_metric,
)
                              
gc.collect()

In [ ]:
# Optional: rotate trajectory direction by 180-deg
adata.obs['t'] = adata.obs['t'].max() - adata.obs['t']
adata.obs['milestones'] = adata.obs['milestones'].astype(int).max() - adata.obs['milestones'].astype(int)
adata.obs['milestones'] = adata.obs['milestones'].astype('category')

adata_desi.obs['t'] = adata_desi.obs['t'].max() - adata_desi.obs['t']
adata_desi.obs['milestones'] = adata_desi.obs['milestones'].astype(int).max() - adata_desi.obs['milestones'].astype(int)
adata_desi.obs['milestones'] = adata_desi.obs['milestones'].astype('category')

In [ ]:
scf.pl.graph(adata, basis="umap")

In [ ]:
plot.disp_trajectory(adata, title='Spatial Gradients\n (LYNX)')

sq.pl.spatial_scatter(
    adata, color='t', 
    title='Spatial Gradients\n (LYNX)',
    cmap='RdBu_r', size=20, img=False
)

sq.pl.spatial_scatter(
    adata, color='milestones', 
    cmap='tab10', size=20, img=False
)

#### Test Associations (fitted expressions along the trajectory)

In [ ]:
# PV-tip as root
scf.tl.test_association(adata, n_jobs=20)
scf.tl.test_association(adata, reapply_filters=True, A_cut=0.1)
scf.tl.root(adata, n_nodes+1) 
scf.tl.fit(adata, n_jobs=20)

scf.tl.test_association(adata_desi, n_jobs=20)
scf.tl.test_association(adata_desi, reapply_filters=True, A_cut=0.1)
scf.tl.root(adata_desi, 7)
scf.tl.fit(adata_desi, n_jobs=20)

gc.collect()

**Notes**<br>Significant DEGs: directly subsetting adata<br>
Fitted DEG expressions: `adata.layers['fitted']`

In [ ]:
sorted_genes = scf.pl.trends(
    adata, 
    features=adata.var_names,
    highlight_features='fdr',
    ordering='max',
    n_features=30,
    fontsize=2.5,
    plot_emb=False,
    return_genes=True
)

sorted_metabolites = scf.pl.trends(
    adata_desi, 
    features=adata_desi.var_names,
    highlight_features='fdr',
    ordering='max',
    n_features=30,
    fontsize=2.5,
    plot_emb=False,
    return_genes=True
)

In [ ]:
# Check spatial distributions of CV-close genes, why so biased towwards PV?
marker_genes = ['MYC', 'KRT7', 'HPX', 'KNG1', 'NAT8', 'LGR5']
sq.pl.spatial_scatter(
    adata, color=marker_genes, ncols=2,
    cmap='magma', size=20, img=False
)

marker_metabolites = ['PE(P-18:0/20:4)', '863.73591 m/z ± 50 ppm',
                      'PI 18:0/18:2', '769.50513 m/z ± 50 ppm',
                      'PE 18:0/20:5', 'PI 18:0/20:4']
sq.pl.spatial_scatter(
    adata_desi, color=marker_metabolites, ncols=2,
    cmap='magma', size=1, img=False
)

In [ ]:
sorted_cells = adata.obs['t'].sort_values().index

fitted_gexp_df = pd.DataFrame(
    adata.layers['fitted'],
    index=adata.obs_names,
    columns=adata.var_names,
)

fitted_gexp_df = fitted_gexp_df.loc[sorted_cells, sorted_genes]
fitted_gexp_df.to_csv('../results/{}_fitted_genes.csv'.format(sample_id), index=True)

fitted_mexp_df = pd.DataFrame(
    adata_desi.layers['fitted'],
    index=adata_desi.obs_names,
    columns=adata_desi.var_names,
)
fitted_mexp_df = fitted_mexp_df.loc[sorted_cells, sorted_metabolites]
fitted_mexp_df.to_csv('../results/{}_fitted_metabolites.csv'.format(sample_id), index=True)

In [ ]:
# Load differentially expressed features
sample_id = 'NIH_F5'
fitted_gexp_df = pd.read_csv('../results/{}_fitted_genes.csv'.format(sample_id), index_col=[0])
fitted_mexp_df = pd.read_csv('../results/{}_fitted_metabolites.csv'.format(sample_id), index_col=[0])


In [ ]:
binned_gexp_df = plot.disp_fitted_expr(
    fitted_gexp_df, nbins=500, 
    return_expr=True,
    savedir='../results/gene_interp_{}'.format(sample_id)
)
binned_gexp_df.to_csv('../results/gene_interp_{}.csv'.format(sample_id), index=True)

binned_mexp_df = plot.disp_fitted_expr(
    fitted_mexp_df, nbins=500, 
    return_expr=True,
    savedir='../results/metabolite_interp_{}'.format(sample_id)
)
binned_mexp_df.to_csv('../results/metabolite_interp_{}.csv'.format(sample_id), index=True)


In [ ]:
sample_fig.write_html("../results/gene_interp_{}.html".format(sample_id))

In [ ]:
# Archived: fitted expressions of specific feature / feature groups

# # Xenium significant features
# scf.pl.single_trend(adata, feature='ADH1C', basis='umap', 
#                     alpha_expr=0.1, size_expr=0.01)
# scf.pl.single_trend(adata, feature='HAMP', basis='umap', 
#                     alpha_expr=0.1, size_expr=0.01)
# scf.pl.single_trend(adata, feature='DPT', basis='umap', 
#                     alpha_expr=0.1, size_expr=0.01)


# # DESI significant features
# scf.pl.single_trend(adata_desi, feature='FA 22:5', basis='umap', 
#                     alpha_expr=0.1, size_expr=0.01)
# scf.pl.single_trend(adata_desi, feature='PA (38:4)', basis='umap', 
#                     alpha_expr=0.1, size_expr=0.01)
# scf.pl.single_trend(adata_desi, feature='AMP', basis='umap', 
#                     alpha_expr=0.1, size_expr=0.01)

# gc.collect()


# FA_markers = [label for label in adata_desi.var_names if 'FA' in label]
# PE_markers = [label for label in adata_desi.var_names if 'PE' in label]
# PI_markers = [label for label in adata_desi.var_names if 'PI' in label]
# PS_markers = [label for label in adata_desi.var_names if 'PS' in label]

# scf.pl.matrix(adata_desi, FA_markers, nbins=100,
#               figsize=(8, len(FA_markers)//2),
#               norm='minmax', annot_top=False, cmap='RdBu_r')

# scf.pl.matrix(adata_desi, PE_markers, nbins=100,
#               figsize=(8, len(PE_markers)//2),
#               norm='minmax', annot_top=False, cmap='RdBu_r')

# scf.pl.matrix(adata_desi, PI_markers, nbins=100,
#               figsize=(8, len(PI_markers)//2),
#               norm='minmax', annot_top=False, cmap='RdBu_r')

# scf.pl.matrix(adata_desi, PS_markers, nbins=100,
#               figsize=(8, len(PS_markers)//2),
#               norm='minmax', annot_top=False, cmap='RdBu_r')


#### Cell-type dynamics

- (1). Cell type compositions & (2) co-localization dynamics along the trajectory

In [ ]:
# Load cell-type annotation
annots = IO.load_xenium(os.path.join(xenium_path, sample_id), raw_count=False).obs['cell_type']
annots = annots.loc[adata.obs_names]

dynamics_df = utils.get_dynamics(adata, annots)
disp_dynamics(
    dynamics_df, 
    # dynamics_df[['PP_Hep', 'M2 Macrophage', 'Cholangiocyte', 'Sinusoidal', 'PC_Hep']],
    ncol=4,
    # savedir='../results/plots/cell_dynamics.pdf'
)

dynamics_df.to_csv('../results/plots/dynamics_{}.csv'.format(sample_id), index=True)

#### Xenium - DESI correlations

In [ ]:
fitted_xenium = adata.layers['fitted']  # [N x G]
fitted_desi = adata_desi.layers['fitted']  # [N x M]
fitted_concat = np.hstack((fitted_xenium, fitted_desi))
fitted_concat.shape

In [ ]:
gene_corr = np.corrcoef(fitted_xenium.T)
gene_corr_df = pd.DataFrame(
    gene_corr,
    index=list(adata.var_names),
    columns=list(adata.var_names)
)

metabolite_corr = np.corrcoef(fitted_desi.T)
metabolite_corr_df = pd.DataFrame(
    metabolite_corr,
    index=list(adata_desi.var_names),
    columns=list(adata_desi.var_names)
)
    
cross_corr = np.corrcoef(fitted_concat.T)
cross_corr_df = pd.DataFrame(
    cross_corr[:adata.shape[1], adata.shape[1]:], 
    index=list(adata.var_names),
    columns=list(adata_desi.var_names)
)

In [ ]:
# Save tentative heatmaps
gene_corr_df.to_csv('../results/xenium_corr_{}.csv'.format(sample_id), index=True)
metabolite_corr_df.to_csv('../results/DESI_corr_{}.csv'.format(sample_id), index=True)
cross_corr_df.to_csv('../results/xenium_DESI_corr_{}.csv'.format(sample_id), index=True)

In [ ]:
### Extract the clustered data
import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go
from plotly.subplots import make_subplots

g = sns.clustermap(metabolite_corr_df, cmap='RdBu_r')
clustered_data = g.data2d

### Create a dendrogram for the rows
dendro_row = ff.create_dendrogram(clustered_data.values, orientation='left', labels=clustered_data.index)
for trace in dendro_row['data']:
    trace['yaxis'] = 'y2'  # Linking this dendrogram to a different y-axis

### Create a dendrogram for the columns
dendro_col = ff.create_dendrogram(clustered_data.values.T, orientation='bottom', labels=clustered_data.columns)
for trace in dendro_col['data']:
    trace['xaxis'] = 'x2'  # Linking this dendrogram to a different x-axis

### Create main heatmap
heatmap = go.Heatmap(
    z=clustered_data.values,
    x=clustered_data.columns,
    y=clustered_data.index,
    colorscale='RdBu_r'
)

In [ ]:
### Combine Dendrograms and Heatmap into a Single Plotly Figure
fig = make_subplots(
    rows=2, cols=2,
    row_heights=[0.2, 0.8], column_widths=[0.8, 0.2],
    specs=[[{"type": "xy"}, None],
           [{"type": "heatmap"}, {"type": "xy"}]],
    horizontal_spacing=1, vertical_spacing=0.1
)

# Add the column dendrogram (top side)
for trace in dendro_col['data']:
    fig.add_trace(trace, row=1, col=1)

# Add the row dendrogram (right side)
for trace in dendro_row['data']:
    fig.add_trace(trace, row=2, col=2)

# Add the heatmap (bottom-left corner)
fig.add_trace(heatmap, row=2, col=1)

### Layout
# Update specific axes settings
fig.update_xaxes(domain=[0, 0.8], row=2, col=1)  # Heatmap X-axis
fig.update_yaxes(domain=[0.04, 0.76], row=2, col=1)  # Heatmap Y-axis

# Column dendrogram
fig.update_xaxes(domain=[0, 0.8], showticklabels=False, row=1, col=1)  
fig.update_yaxes(domain=[0.78, 1], showticklabels=False, row=1, col=1)  

# Row dendrogram
fig.update_xaxes(domain=[0.82, 1], showticklabels=False, row=2, col=2) 
fig.update_yaxes(domain=[0, 0.8], showticklabels=False, row=2, col=2) 

### Final layout adjustments
fig.update_layout(
    height=800, width=800,
    showlegend=False,
    hovermode='closest',
    plot_bgcolor='white',
    margin=dict(l=100, r=50, t=50, b=100),
)


In [ ]:
# fig.write_html("../results/Xenium_heatmap_{}.html".format(sample_id))
fig.write_html("../results/DESI_heatmap_{}.html".format(sample_id))
# fig.write_html("../results/Xenium_DESI_heatmap_{}.html".format(sample_id))

gc.collect()

#### All-samples

In [ ]:
xenium_path = '../data/xenium/'
desi_path = '../data/integrated/desi_xenium_map/'
sample_ids = sorted([
    sample_id for sample_id in os.listdir(xenium_path)
    if os.path.isdir(os.path.join(xenium_path, sample_id))
])


In [ ]:
from importlib import reload
reload(plot)
reload(utils)

In [ ]:
n_latent = 6
n_nodes = 10  # number of nodes for principal graph constrcution
n_bins = 500

cutoff_thresh = 0.1  # significance level for Test association 
dist_metric = 'euclidean'

outdir = '../results/downstream/'
if not os.path.isdir(outdir):
    os.makedirs(outdir, exist_ok=True)

for sample_id in sample_ids:
    print('Downstream analysis for {}'.format(sample_id))
    
    #-------------
    # Load data
    #-------------
    latent = np.load('../results/lynx_{0}_{1}.npy'.format(n_latent, sample_id))

    adata = IO.load_xenium(os.path.join(xenium_path, sample_id)) 
    adata_desi = IO.load_desi(os.path.join(desi_path, sample_id+'.h5ad'), raw_img=False)
    adata, adata_desi = IO.filter_cells(adata, adata_desi)

    adata.obsm['X_z'] = latent
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)
    sc.pp.neighbors(adata, use_rep='X_z')
    sc.tl.umap(adata)

    adata_desi.obsm['X_z'] = latent
    adata_desi.obsm['X_umap'] = adata.obsm['X_umap']

    #---------------------
    # Compute trajectory
    #---------------------
    trajectory.compute_trajectory(
        adata, 
        use_rep='X_z',
        n_nodes=n_nodes,
        dist_metric=dist_metric,
    )

    trajectory.compute_trajectory(
        adata_desi,
        use_rep='X_z',
        n_nodes=n_nodes,
        dist_metric=dist_metric,
    )

    # Optional: rotate trajectory direction by 180-deg
    adata.obs['t'] = adata.obs['t'].max() - adata.obs['t']
    adata.obs['milestones'] = adata.obs['milestones'].astype(int).max() - adata.obs['milestones'].astype(int)
    adata.obs['milestones'] = adata.obs['milestones'].astype('category')

    adata_desi.obs['t'] = adata_desi.obs['t'].max() - adata_desi.obs['t']
    adata_desi.obs['milestones'] = adata_desi.obs['milestones'].astype(int).max() - adata_desi.obs['milestones'].astype(int)
    adata_desi.obs['milestones'] = adata_desi.obs['milestones'].astype('category')

    #---------------------------------------------------------------
    # Statistical tests & fitting expressions along the trajectory
    #---------------------------------------------------------------
    scf.tl.test_association(adata, n_jobs=20)
    scf.tl.test_association(adata, reapply_filters=True, A_cut=0.1)
    scf.tl.root(adata, n_nodes+1)  # "PV"-tip as root
    scf.tl.fit(adata,n_jobs=20)

    scf.tl.test_association(adata_desi, n_jobs=20)
    scf.tl.test_association(adata_desi, reapply_filters=True, A_cut=0.1)
    scf.tl.root(adata_desi, n_nodes+1)
    scf.tl.fit(adata_desi, n_jobs=20)

    # Extract fitted features sorted along trajectory
    fitted_gexp_df = utils.sort_fitted_expr(adata)    
    binned_gexp_df = plot.disp_fitted_expr(
        fitted_gexp_df, nbins=n_bins, return_expr=True,
        savedir=os.path.join(outdir, 'gene_interp_{}'.format(sample_id))
    )

    fitted_gexp_df.to_csv(os.path.join(
        outdir,
        'fitted_genes.csv_{}'.format(sample_id)
    ), index=True)
    # binned_gexp_df.to_csv(os.path.join(outdir, 'gene_interp_{}.csv'.format(sample_id)), index=True)

    fitted_mexp_df = utils.sort_fitted_expr(adata_desi)    
    binned_mexp_df = plot.disp_fitted_expr(
        fitted_mexp_df, nbins=n_bins, return_expr=True,
        savedir=os.path.join(outdir, 'metabolite_interp_{}'.format(sample_id))
    )
    
    fitted_mexp_df.to_csv(os.path.join(
        outdir,
        'fitted_metabolites_{}.csv'.format(sample_id)
    ), index=True)
    # binned_mexp_df.to_csv(os.path.join(outdir, 'metabolite_interp_{}.csv)'.format(sample_id)), index=True)

    del fitted_gexp_df, fitted_mexp_df, binned_gexp_df, binned_mexp_df
    gc.collect()

    #-----------------------------
    # Compute cell-type dynamics
    #-----------------------------
    annots = IO.load_xenium(os.path.join(xenium_path, sample_id), raw_count=False).obs['cell_type']
    annots = annots.loc[adata.obs_names]

    dynamics_df = utils.get_dynamics(adata, annots)
    plot.disp_dynamics(
        dynamics_df, ncols=4, 
        savedir=os.path.join(outdir, 'dynamics_{}.pdf'.format(sample_id))
    )

    dynamics_df.to_csv(os.path.join(outdir, 'dynamics_{}.csv'.format(sample_id)))

    del dynamics_df
    del adata, adata_desi, annots
    gc.collect()

    print('===================\n\n')


---